# Day 10 — Dictionaries & JSON
**Xeven Solutions AI Engineering Internship**  
**Author:** Sehar Andleeb | **Mentor:** Mubashir Sir  

---
## Learning Map
| # | Concept | AI Relevance |
|---|---------|-------------|
| 1 | Dict fundamentals & O(1) lookup | Token vocab lookup in LLMs |
| 2 | Dict methods | Config parsing, feature extraction |
| 3 | Nested dicts | Dataset metadata, model registries |
| 4 | JSON read/write | Model configs, pipeline checkpoints |
| 5 | Dict comprehensions | Feature engineering transforms |
| 6 | Task 1 — Student System | Nested data + JSON persistence |
| 7 | Task 2 — Inventory Manager | Aggregation + category analytics |
| 8 | Task 3 — Config Manager | Real-world AI pipeline pattern |

---
## Part 1 — Dictionary Fundamentals

In [1]:
# ── 1.1  Creating dictionaries ──────────────────────────────────────────────

# Literal syntax
model_info = {
    "name": "gpt-4o",
    "provider": "OpenAI",
    "context_window": 128_000,
    "supports_vision": True,
}

# dict() constructor
hyperparams = dict(learning_rate=3e-4, batch_size=32, epochs=10)

# From a list of (key, value) pairs — common when parsing CSV headers
labels = dict([("cat", 0), ("dog", 1), ("bird", 2)])

print("Model info  :", model_info)
print("Hyperparams :", hyperparams)
print("Labels      :", labels)

Model info  : {'name': 'gpt-4o', 'provider': 'OpenAI', 'context_window': 128000, 'supports_vision': True}
Hyperparams : {'learning_rate': 0.0003, 'batch_size': 32, 'epochs': 10}
Labels      : {'cat': 0, 'dog': 1, 'bird': 2}


In [2]:
# ── 1.2  CRUD operations ────────────────────────────────────────────────────

# Create / Update
model_info["max_tokens"] = 4096             # add new key
model_info["context_window"] = 200_000      # update existing key

# Read — direct access raises KeyError if missing
print("Name      :", model_info["name"])

# Safe read — .get() returns default instead of KeyError
temperature = model_info.get("temperature", 0.7)
print("Temp (default) :", temperature)

# Delete
removed = model_info.pop("supports_vision", None)   # None = safe pop
print("Removed 'supports_vision' :", removed)
print("\nUpdated model_info:\n", model_info)

Name      : gpt-4o
Temp (default) : 0.7
Removed 'supports_vision' : True

Updated model_info:
 {'name': 'gpt-4o', 'provider': 'OpenAI', 'context_window': 200000, 'max_tokens': 4096}


---
## Part 2 — Dictionary Methods

In [3]:
# ── 2.1  Iteration helpers ──────────────────────────────────────────────────

token_vocab = {"hello": 0, "world": 1, "python": 2, "AI": 3, "Xeven": 4}

print("Keys   :", list(token_vocab.keys()))
print("Values :", list(token_vocab.values()))
print("Items  :", list(token_vocab.items()))

print("\n--- iterating with .items() ---")
for token, token_id in token_vocab.items():
    print(f"  '{token}' → token_id {token_id}")

Keys   : ['hello', 'world', 'python', 'AI', 'Xeven']
Values : [0, 1, 2, 3, 4]
Items  : [('hello', 0), ('world', 1), ('python', 2), ('AI', 3), ('Xeven', 4)]

--- iterating with .items() ---
  'hello' → token_id 0
  'world' → token_id 1
  'python' → token_id 2
  'AI' → token_id 3
  'Xeven' → token_id 4


In [4]:
# ── 2.2  Merge and update ────────────────────────────────────────────────────

base_config = {"batch_size": 32, "lr": 1e-3, "epochs": 5}
experiment_overrides = {"lr": 3e-4, "dropout": 0.1}   # override lr, add dropout

# .update() modifies in-place — later dict wins on collision
merged = base_config.copy()         # don't mutate original
merged.update(experiment_overrides)
print("Merged config:", merged)

# Python 3.9+ unpacking merge (immutable, creates new dict)
merged_v2 = {**base_config, **experiment_overrides}
print("Merged v2    :", merged_v2)

Merged config: {'batch_size': 32, 'lr': 0.0003, 'epochs': 5, 'dropout': 0.1}
Merged v2    : {'batch_size': 32, 'lr': 0.0003, 'epochs': 5, 'dropout': 0.1}


---
## Part 3 — Nested Dictionaries

In [5]:
# ── 3.1  Nested structure ───────────────────────────────────────────────────

# AI use-case: each sample in a dataset as a nested dict
dataset = {
    "sample_001": {
        "text": "The model achieved 95% accuracy.",
        "label": "positive",
        "metadata": {"source": "arxiv", "year": 2024, "tokens": 7},
    },
    "sample_002": {
        "text": "Training failed due to OOM error.",
        "label": "negative",
        "metadata": {"source": "blog", "year": 2023, "tokens": 7},
    },
}

# Chained key access
print("Sample 001 label  :", dataset["sample_001"]["label"])
print("Sample 002 source :", dataset["sample_002"]["metadata"]["source"])

# Safe nested access with chained .get()
year = dataset.get("sample_003", {}).get("metadata", {}).get("year", "unknown")
print("Missing sample year (safe) :", year)

Sample 001 label  : positive
Sample 002 source : blog
Missing sample year (safe) : unknown


In [6]:
# ── 3.2  Updating nested values ─────────────────────────────────────────────

# Add a new field deep inside the structure
dataset["sample_001"]["metadata"]["reviewed"] = True

# Add an entirely new sample
dataset["sample_003"] = {
    "text": "Transformer attention is O(n²) in sequence length.",
    "label": "neutral",
    "metadata": {"source": "textbook", "year": 2025, "tokens": 9},
}

print(f"Dataset now has {len(dataset)} samples.")
for sid, sample in dataset.items():
    print(f"  {sid}: [{sample['label']}] {sample['text'][:45]}...")

Dataset now has 3 samples.
  sample_001: [positive] The model achieved 95% accuracy....
  sample_002: [negative] Training failed due to OOM error....
  sample_003: [neutral] Transformer attention is O(n²) in sequence le...


---
## Part 4 — JSON Read / Write

In [7]:
import json
import os

# ── 4.1  json.dump() — write to file ────────────────────────────────────────
output_file = "dataset_snapshot.json"

with open(output_file, "w", encoding="utf-8") as fh:
    json.dump(dataset, fh, indent=4, ensure_ascii=False)

print(f"Written to '{output_file}'.")
print("File size:", os.path.getsize(output_file), "bytes")

Written to 'dataset_snapshot.json'.
File size: 745 bytes


In [8]:
# ── 4.2  json.load() — read from file ───────────────────────────────────────
with open(output_file, "r", encoding="utf-8") as fh:
    loaded_dataset = json.load(fh)

print("Loaded dataset:")
print(json.dumps(loaded_dataset, indent=2))  # json.dumps → string (for printing)

Loaded dataset:
{
  "sample_001": {
    "text": "The model achieved 95% accuracy.",
    "label": "positive",
    "metadata": {
      "source": "arxiv",
      "year": 2024,
      "tokens": 7,
      "reviewed": true
    }
  },
  "sample_002": {
    "text": "Training failed due to OOM error.",
    "label": "negative",
    "metadata": {
      "source": "blog",
      "year": 2023,
      "tokens": 7
    }
  },
  "sample_003": {
    "text": "Transformer attention is O(n\u00b2) in sequence length.",
    "label": "neutral",
    "metadata": {
      "source": "textbook",
      "year": 2025,
      "tokens": 9
    }
  }
}


In [9]:
# ── 4.3  json.loads() / json.dumps() — strings, not files ───────────────────
json_string = '{"model": "claude-3-5-sonnet", "temperature": 0.5}'
parsed = json.loads(json_string)             # string → dict
back_to_string = json.dumps(parsed, indent=2)  # dict → string

print("Parsed type :", type(parsed))
print("Dict        :", parsed)
print("Back to str :\n", back_to_string)

Parsed type : <class 'dict'>
Dict        : {'model': 'claude-3-5-sonnet', 'temperature': 0.5}
Back to str :
 {
  "model": "claude-3-5-sonnet",
  "temperature": 0.5
}


---
## Part 5 — Dictionary Comprehensions

In [10]:
# ── 5.1  Basic comprehension: {k: v for k, v in pairs} ──────────────────────

# Reverse a vocabulary (token → id becomes id → token)
vocab = {"hello": 0, "world": 1, "python": 2, "AI": 3}
id_to_token = {token_id: token for token, token_id in vocab.items()}
print("id_to_token:", id_to_token)

# Filter — only keep high-frequency tokens (simulated)
token_freqs = {"the": 5000, "a": 4200, "model": 320, "transformer": 180, "xeven": 12}
common_tokens = {t: f for t, f in token_freqs.items() if f >= 200}
print("Common tokens (freq ≥ 200):", common_tokens)

id_to_token: {0: 'hello', 1: 'world', 2: 'python', 3: 'AI'}
Common tokens (freq ≥ 200): {'the': 5000, 'a': 4200, 'model': 320}


In [11]:
# ── 5.2  Value transformation ────────────────────────────────────────────────

# Normalise scores to 0-1 range (min-max scaling) — AI preprocessing step
raw_scores = {"math": 78, "physics": 92, "cs": 88, "english": 65}
min_s, max_s = min(raw_scores.values()), max(raw_scores.values())
normalised = {
    subject: round((score - min_s) / (max_s - min_s), 3)
    for subject, score in raw_scores.items()
}
print("Normalised scores:", normalised)

# Nested comprehension — flatten a list of tags per sample
tag_data = [
    ("doc1", ["nlp", "transformer"]),
    ("doc2", ["vision", "cnn"]),
    ("doc3", ["nlp", "bert"]),
]
doc_tags = {doc_id: tags for doc_id, tags in tag_data}
print("Doc tags:", doc_tags)

Normalised scores: {'math': 0.481, 'physics': 1.0, 'cs': 0.852, 'english': 0.0}
Doc tags: {'doc1': ['nlp', 'transformer'], 'doc2': ['vision', 'cnn'], 'doc3': ['nlp', 'bert']}


---
## Part 6 — Task 1: Student Information System
*(Imports the standalone script — run it first or cells will auto-handle)*

In [12]:
import sys, os
sys.path.insert(0, os.getcwd())   # ensure local scripts are importable

from student_information_system import (
    add_student, update_grade, get_student_average,
    find_top_student, generate_report, save_students,
)

# Fresh in-memory registry
students = {}

add_student(students, "NB01", "Nadia Baig", 21, {
    "Python": 94, "AI Fundamentals": 91, "Data Structures": 89,
})
add_student(students, "ZK02", "Zain Khan", 22, {
    "Python": 77, "AI Fundamentals": 82, "Data Structures": 71,
})
add_student(students, "RM03", "Rida Malik", 20)

# Add grades for Rida
update_grade(students, "RM03", "Python", 88)
update_grade(students, "RM03", "AI Fundamentals", 79)

print("\n--- Individual averages ---")
for sid in students:
    avg = get_student_average(students, sid)
    print(f"  {students[sid]['name']}: {avg}")

print("\n--- Top student ---")
top = find_top_student(students)
print(f"  {top[1]} with avg {top[2]}")

generate_report(students)
save_students(students, "notebook_students.json")

[INFO] Added student: Nadia Baig (ID: NB01).
[INFO] Added student: Zain Khan (ID: ZK02).
[INFO] Added student: Rida Malik (ID: RM03).
[INFO] Updated Rida Malik — Python: 88.
[INFO] Updated Rida Malik — AI Fundamentals: 79.

--- Individual averages ---
  Nadia Baig: 91.33
  Zain Khan: 76.67
  Rida Malik: 83.5

--- Top student ---
  Nadia Baig with avg 91.33

                   STUDENT PERFORMANCE REPORT                    
Rank  Name                 ID           GPA  Grade
-----------------------------------------------------------------
1     Nadia Baig           NB01       91.33      A  ✓
2     Rida Malik           RM03       83.50     B+  ✓
3     Zain Khan            ZK02       76.67      B  ✓

-----------------------------------------------------------------
SUBJECT BREAKDOWN
-----------------------------------------------------------------

  Nadia Baig (Age 21):
    AI Fundamentals        91.0  █████████
    Data Structures        89.0  ████████
    Python                 94.0  ██

True

---
## Part 7 — Task 2: Product Inventory Manager

In [13]:
from product_inventory_manager import (
    add_product, update_stock, search_by_category,
    low_stock_alert, calculate_total_inventory_value,
    average_price_per_category, print_inventory_summary,
    export_inventory_report,
)

inventory = {}

add_product(inventory, "P001", "RTX 4090",        350_000, 4,  "GPU")
add_product(inventory, "P002", "A100 80GB",      1_200_000, 2,  "GPU")
add_product(inventory, "P003", "NVMe SSD 4TB",     55_000, 20, "Storage")
add_product(inventory, "P004", "64GB DDR5 RAM",    42_000, 8,  "RAM")
add_product(inventory, "P005", "10GbE NIC",        18_000, 5,  "Networking")
add_product(inventory, "P006", "UPS 3000VA",       35_000, 3,  "Power")

update_stock(inventory, "P001", -1)   # one GPU sold
update_stock(inventory, "P003", 10)   # restock SSDs

# Category search
gpus = search_by_category(inventory, "GPU")
print("\nGPU products:")
for pid, info in gpus.items():
    print(f"  {pid}: {info['name']} — PKR {info['price']:,} (qty {info['quantity']})")

# Low stock
alerts = low_stock_alert(inventory, threshold=5)
print(f"\n⚠  Low stock items (≤5): {len(alerts)}")
for pid, name, qty in alerts:
    print(f"  {pid} | {name} | qty: {qty}")

print(f"\nTotal inventory value: PKR {calculate_total_inventory_value(inventory):,}")
print("Avg price per category:", average_price_per_category(inventory))

print_inventory_summary(inventory)
export_inventory_report(inventory, "notebook_inventory_report.json")

[INFO] Added product: RTX 4090 (ID: P001, Category: GPU).
[INFO] Added product: A100 80GB (ID: P002, Category: GPU).
[INFO] Added product: NVMe SSD 4TB (ID: P003, Category: Storage).
[INFO] Added product: 64GB DDR5 RAM (ID: P004, Category: RAM).
[INFO] Added product: 10GbE NIC (ID: P005, Category: Networking).
[INFO] Added product: UPS 3000VA (ID: P006, Category: Power).
[INFO] Sold 1 unit(s) of 'RTX 4090'. New stock: 3.
[INFO] Restocked 10 unit(s) of 'NVMe SSD 4TB'. New stock: 30.
[INFO] Found 2 product(s) in category 'GPU'.

GPU products:
  P001: RTX 4090 — PKR 350,000 (qty 3)
  P002: A100 80GB — PKR 1,200,000 (qty 2)

⚠  Low stock items (≤5): 4
  P002 | A100 80GB | qty: 2
  P001 | RTX 4090 | qty: 3
  P006 | UPS 3000VA | qty: 3
  P005 | 10GbE NIC | qty: 5

Total inventory value: PKR 5,631,000
Avg price per category: {'GPU': 775000.0, 'Storage': 55000.0, 'RAM': 42000.0, 'Networking': 18000.0, 'Power': 35000.0}

                      PRODUCT INVENTORY SUMMARY                       
ID 

True

---
## Part 8 — Task 3: Configuration Manager

In [14]:
from configuration_manager import (
    load_config, save_config, validate_config,
    get_value, get_feature_flag, update_config_value,
    switch_environment, print_config_summary,
    DEFAULT_CONFIG,
)
from copy import deepcopy

# Work from a copy so we don't clobber the real config.json
config = deepcopy(DEFAULT_CONFIG)

# Validation
is_valid, issues = validate_config(config)
print(f"Valid: {is_valid} | Issues: {issues or 'none'}")

# Safe access
print("\nDB port       :", get_value(config, "database", "port", 5432))
print("Caching on?   :", get_feature_flag(config, "enable_caching"))
print("Missing key?  :", get_value(config, "database", "replica_host", "not configured"))

# Programmatic updates
update_config_value(config, "model", "temperature", 0.2)
update_config_value(config, "model", "max_tokens", 8192)

# Switch to production
switch_environment(config, "production")

print_config_summary(config)

save_config(config, "notebook_config.json")
print("Config saved to notebook_config.json")

Valid: True | Issues: none

DB port       : 5432
Caching on?   : True
Missing key?  : not configured
[INFO] config['model']['temperature'] updated: 0.7 → 0.2.
[INFO] config['model']['max_tokens'] updated: 2048 → 8192.
[INFO] config['app']['environment'] updated: 'development' → 'production'.
[INFO] config['app']['debug'] updated: True → False.
[INFO] config['app']['log_level'] updated: 'INFO' → 'WARNING'.
[INFO] config['features']['enable_metrics'] updated: False → True.
[INFO] Production presets applied (debug=False, metrics=True).

                 CONFIGURATION SUMMARY                 

  [APP]
    name                      : XevenAI Platform
    version                   : 1.0.0
    environment               : production
    debug                     : False
    log_level                 : WARNING

  [DATABASE]
    host                      : localhost
    port                      : 5432
    name                      : xeven_ai_db
    user                      : admin
    password

---
## Part 9 — Key Takeaways & AI Connections

| Today's Concept | Where you'll use it in AI Engineering |
|-----------------|---------------------------------------|
| `dict[key]` O(1) lookup | Token ID lookup in LLM vocabulary tables |
| `.get(key, default)` | Safe config access in training scripts |
| Nested dicts | Dataset manifests, model metadata stores |
| `json.load/dump` | Saving model configs, experiment results |
| Dict comprehensions | Feature engineering, label encoding |
| `defaultdict` | Accumulating metrics per class/category |
| Config manager pattern | Every production ML pipeline uses this |

```
Tomorrow — Day 11: Functions & Modular Programming
"Stop writing scripts. Start writing systems."
                          — Mubashir Sir
```